<a href="https://colab.research.google.com/github/jfemiani/cse620b-public/blob/main/labs/harvard-forest-lai-key.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Worked key · Can satellites tell how leafy a forest is?

Use this notebook after class to check your work. It shows one solution using NDVI and the VH/VV radar ratio. You can follow the same steps with other indices.

[Open the student handout](https://colab.research.google.com/github/jfemiani/cse620b-public/blob/main/labs/harvard-forest-lai.ipynb)

## 1. Download the data

The ZIP contains small satellite images with the original bands on a common 20 m grid. The CSV contains the field measurements.

In [ ]:
%%bash
set -e
wget -nv -O harvard-forest-chips.zip https://vision.csi.miamioh.edu/data/cse620b/forestry/harvard-forest-chips.zip
unzip -oq harvard-forest-chips.zip -d harvard-forest
wget -nv -O lai.csv https://harvardforest1.fas.harvard.edu/data/p06/hf069/hf069-02-lai-site.csv

## 2. Find the study area

These color views show the same 1.2 km-wide area at Harvard Forest's Prospect Hill tract in Massachusetts. The white outline marks the approximate field survey area, where transects run west of the Environmental Measurement Site (EMS) tower. It covers roughly the western half of a 500 m radius around the tower.

**Look first.** Where does the forest change most between June and November? Which changes might a radar image pick up?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

footprint = np.load("harvard-forest/footprint.npy")

def load_bands(visit):
    with np.load(f"harvard-forest/{visit}.npz") as chip:
        return dict(chip)

def show_rgb(ax, bands):
    rgb = np.dstack([bands["B04"], bands["B03"], bands["B02"]])
    ax.imshow(np.clip(rgb * 4, 0, 1))
    ax.contour(footprint, levels=[0.5], colors="white", linewidths=1)
    ax.set_axis_off()

fig, axes = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
for ax, visit, label in zip(
    axes,
    ["2023-06-05", "2023-11-07"],
    ["June 11, 2023", "November 8, 2023"],
):
    show_rgb(ax, load_bands(visit))
    ax.set_title(label)
plt.show()

## 3. Read and display the source bands

Each band is a 2D array. Sentinel-2 values are surface reflectance. Sentinel-1 values are linear radar backscatter power. This radar series has **VV** and **VH**. The letters give the transmit and receive polarizations, respectively. **HH** (horizontal transmit and receive) and **HV** are other possible channels. The archive does not carry them.

Run the cell to see each band and the values of one pixel. The color scale is set separately for each band. Use its colorbar when comparing values. Try changing the visit to `"2023-11-07"`. What changes in red and near-infrared? What changes in VV and VH?

In [ ]:
visit = "2023-06-05"  # Date of the field visit used to name the image file
bands = load_bands(visit)
band_names = {
    "Red (B04)": "B04", "Green (B03)": "B03", "Blue (B02)": "B02",
    "NIR (B08)": "B08", "SWIR 1 (B11)": "B11", "SWIR 2 (B12)": "B12",
    "VV": "VV", "VH": "VH",
}

fig, axes = plt.subplots(2, 4, figsize=(13, 6), layout="constrained")
for ax, (label, band) in zip(axes.flat, band_names.items()):
    image = ax.imshow(bands[band], cmap="gray")
    ax.contour(footprint, levels=[0.5], colors="cyan", linewidths=0.8)
    ax.set_title(label)
    ax.set_axis_off()
    fig.colorbar(image, ax=ax, shrink=0.7)
plt.show()

row, col = 30, 20
print(f"Pixel at row {row}, column {col}")
for label, band in band_names.items():
    print(f"{label}: {bands[band][row, col]:.4f}")

## 4. Match satellite images to field measurements

**Leaf area index (LAI)** is leaf area per unit of ground area. The table gives the mean LAI across the plots measured on each field visit and the dates of the paired optical and radar images. A field visit gives you one LAI value. To compare it with an index map, you will need to reduce the pixels to one index value for that visit.

Find the June and November visits. Do the measured LAI values agree with your impression from the color images?

In [ ]:
field = pd.read_csv("lai.csv")
field = field.loc[field["site"] == "ems", ["date", "n_plots", "all_mean"]]
field = field.rename(columns={"date": "field_date", "all_mean": "LAI"})
scenes = pd.read_csv("harvard-forest/scenes.csv")
visits = scenes.merge(field, on="field_date", validate="one_to_one")
display(visits[["field_date", "optical_date", "radar_date", "n_plots", "LAI"]])

def usable_pixels(bands):
    # Class 4 means vegetation in the Sentinel-2 scene classification.
    valid = footprint & (bands["SCL"] == 4)
    for band in band_names.values():
        valid &= np.isfinite(bands[band]) & (bands[band] > 0)
    return valid

## 5. Calculate two indices

For NDVI, subtract red from NIR and divide by their sum. For the radar index, divide VH by VV. For each visit, compute both indices over the same usable pixels. The maps show where those pixels are. The median of each map gives one value per visit to match with field LAI.

In [ ]:
def calculate_indices(bands, valid):
    red = bands["B04"][valid]
    nir = bands["B08"][valid]
    vv = bands["VV"][valid]
    vh = bands["VH"][valid]
    return {"NDVI": (nir - red) / (nir + red), "VH/VV": vh / vv}

valid = usable_pixels(bands)
fig, axes = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
for ax, (name, values) in zip(axes, calculate_indices(bands, valid).items()):
    index_map = np.full(valid.shape, np.nan)
    index_map[valid] = values
    image = ax.imshow(index_map)
    ax.set_title(f"{name} · {visit}")
    ax.set_axis_off()
    fig.colorbar(image, ax=ax, shrink=0.75)
plt.show()

summaries = []
for field_date in visits["field_date"]:
    chip = load_bands(field_date)
    values = calculate_indices(chip, usable_pixels(chip))
    summaries.append({name: np.median(pixels) for name, pixels in values.items()})

table = pd.concat([visits, pd.DataFrame(summaries)], axis=1)
display(table[["field_date", "LAI", "NDVI", "VH/VV"]])

## 6. Fit and test the models

Fit the slope and intercept using the 2022–2023 visits. For each 2024 visit, multiply the index by the slope and add the intercept to predict LAI. Compare both models with a baseline that always predicts the 2022–2023 mean LAI.

In [ ]:
is_test = table["field_date"].str.startswith("2024-")
train = table.loc[~is_test]
test = table.loc[is_test]

comparison = test[["field_date", "LAI"]].rename(columns={"LAI": "Measured LAI"})
comparison["Training-mean baseline"] = train["LAI"].mean()

rmse = {}
for name in ("NDVI", "VH/VV"):
    slope, intercept = np.polyfit(train[name], train["LAI"], deg=1)
    comparison[name] = intercept + slope * test[name]
    print(f"{name}: slope = {slope:.3f}, intercept = {intercept:.3f}")

for name in ("NDVI", "VH/VV", "Training-mean baseline"):
    errors = comparison[name] - comparison["Measured LAI"]
    rmse[name] = np.sqrt(np.mean(errors**2))

display(comparison)
display(pd.Series(rmse, name="2024 RMSE").round(3))

## Check your interpretation

Both models beat the training-mean baseline on these three visits. For this pair of indices, the radar ratio has the lower 2024 RMSE. Look at the predictions by date to see where that difference comes from. A comparison across three dates at one site does not establish how the models would perform at another forest.

## Data and worked examples


- [Harvard Forest HF069: Biomass Inventories at the EMS Tower](https://harvardforest1.fas.harvard.edu/exist/apps/datasets/showData.html?id=hf069).
  Uses the `ems` rows of the published site LAI table, not a gridded LAI product.
- Contains modified Copernicus Sentinel data (2022–2024), retrieved through
  [Sentinel-2 L2A](https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a)
  and [Sentinel-1 RTC](https://planetarycomputer.microsoft.com/dataset/sentinel-1-rtc).
- [Rasterio walk-through for Sentinel-2 NDVI](https://www.forestry-ecological-gis.org/ecological-gis-data-foundations-in-python/vegetation-index-calculation-in-python/calculating-ndvi-from-sentinel-2-with-rasterio/),
  including surface-reflectance offsets and masked pixels.
- [Planetary Computer Sentinel-1 RTC tutorial](https://github.com/microsoft/PlanetaryComputerExamples/blob/main/tutorials/customizable-rtc-sentinel1.ipynb).
- [ESA PhiLab radar index examples](https://github.com/ESA-PhiLab/sarpyx/blob/main/docs/user_guide/science_applications.md).